In [1]:
import json
import numpy as np

# Doc feature columns
with open(r'd:\nids-vae-project\artifacts\feature_schema\feature_columns.json', 'r') as f:
    features = json.load(f)
print('=== FEATURES ===')
print(features)
print('So luong:', len(features))

print()

# Doc threshold
with open(r'd:\nids-vae-project\artifacts\threshold\threshold.json', 'r') as f:
    threshold = json.load(f)
print('=== THRESHOLD ===')
print(threshold)

print()

# Doc preprocessing metadata (co the co mean/std o day)
with open(r'd:\nids-vae-project\data\processed\preprocessing_metadata.json', 'r') as f:
    meta = json.load(f)
print('=== PREPROCESSING METADATA ===')
print(json.dumps(meta, indent=2)[:2000])  # in 2000 ky tu dau

=== FEATURES ===
{'feature_columns': ['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'Down/Up Ratio', 'Average Packet Size', 'Avg Fwd Segment Size', '

In [2]:
import joblib
import json

scaler = joblib.load(r'd:\nids-vae-project\artifacts\scaler\scaler.joblib')

print('=== MEAN (10 dau) ===')
print(scaler.mean_[:10].tolist())

print()
print('=== STD (10 dau) ===')
print(scaler.scale_[:10].tolist())

print()
# Doc medians
with open(r'd:\nids-vae-project\artifacts\scaler\imputation_medians.json', 'r') as f:
    medians = json.load(f)
print('=== MEDIANS (10 dau) ===')
items = list(medians.items())[:10]
for k, v in items:
    print(f'{k}: {v}')

=== MEAN (10 dau) ===
[11177.358887598855, 10939160.089081096, 11.846783797289604, 13.414880577979211, 560.5702845891271, 21924.269687168256, 199.33890892998764, 20.08233369548193, 52.06254613693951, 60.32422396164667]

=== STD (10 dau) ===
[21795.760500775923, 29414950.762533374, 1022.8977363501854, 1344.84860417055, 6971.9095852953715, 3066831.893734949, 458.3752042783582, 37.2228767027414, 94.11089739837983, 149.45439844636007]

=== MEDIANS (10 dau) ===
Destination Port: 80.0
Flow Duration: 36816.0
Total Fwd Packets: 2.0
Total Backward Packets: 2.0
Total Length of Fwd Packets: 68.0
Total Length of Bwd Packets: 154.0
Fwd Packet Length Max: 41.0
Fwd Packet Length Min: 6.0
Fwd Packet Length Mean: 38.0
Fwd Packet Length Std: 0.0


In [4]:
import numpy as np
import pandas as pd
import os

os.makedirs(r'd:\nids-vae-project\demo_data', exist_ok=True)

# ── Load du lieu test that ────────────────────────────────────────
X_test = np.load(r'd:\nids-vae-project\data\test\X_test.npy')
y_test = np.load(r'd:\nids-vae-project\data\test\y_test.npy')

with open(r'd:\nids-vae-project\artifacts\feature_schema\feature_columns.json', 'r') as f:
    import json
    schema = json.load(f)
features = schema['feature_columns']

df_test = pd.DataFrame(X_test, columns=features)
df_test['Label_encoded'] = y_test  # 0=BENIGN, 1=ATTACK

print('Tong test set:', len(df_test))
print('BENIGN:', (y_test == 0).sum())
print('ATTACK:', (y_test == 1).sum())
print('Ti le ATTACK:', f"{(y_test==1).mean()*100:.1f}%")

Tong test set: 2019575
BENIGN: 1593697
ATTACK: 425878
Ti le ATTACK: 21.1%


In [ ]:
import numpy as np
import pandas as pd
import joblib
import json
import os

# Tao thu muc neu chua co
os.makedirs(r'd:\nids-vae-project\demo_data2', exist_ok=True)

# Load artifacts
scaler = joblib.load(r'd:\nids-vae-project\artifacts\scaler\scaler.joblib')
with open(r'd:\nids-vae-project\artifacts\scaler\imputation_medians.json', 'r') as f:
    medians = json.load(f)
with open(r'd:\nids-vae-project\artifacts\feature_schema\feature_columns.json', 'r') as f:
    schema = json.load(f)

features = schema['feature_columns']
threshold = 6.0312628746032715

np.random.seed(42)

def make_normal_rows(n):
    median_vals = [medians[f] for f in features]
    rows = []
    for _ in range(n):
        row = {}
        for i, f in enumerate(features):
            noise = np.random.normal(0, 0.1)
            row[f] = max(0, median_vals[i] * (1 + noise))
        rows.append(row)
    return rows

def make_attack_rows(n, intensity=5.0):
    median_vals = [medians[f] for f in features]
    rows = []
    for _ in range(n):
        row = {}
        for i, f in enumerate(features):
            if f in ['Flow Bytes/s', 'Flow Packets/s', 'Fwd Packets/s',
                     'Bwd Packets/s', 'Total Fwd Packets', 'Total Backward Packets',
                     'Flow Duration', 'Packet Length Variance']:
                multiplier = np.random.uniform(intensity, intensity * 3)
                row[f] = median_vals[i] * multiplier
            elif f in ['SYN Flag Count', 'FIN Flag Count', 'PSH Flag Count']:
                row[f] = np.random.randint(1, 5)
            else:
                noise = np.random.normal(0, 0.3)
                row[f] = max(0, median_vals[i] * (1 + noise))
        rows.append(row)
    return rows

# ── DEMO 1: 100 flow binh thuong ─────────────────────────────────
rows1 = make_normal_rows(100)
df1 = pd.DataFrame(rows1, columns=features)
df1['Label'] = 'BENIGN'
df1.to_csv(r'd:\nids-vae-project\demo_data\demo1_normal.csv', index=False)
print("✓ demo1_normal.csv (100 BENIGN)")

# ── DEMO 2: 70 binh thuong + 30 tan cong ─────────────────────────
rows2_normal = make_normal_rows(70)
rows2_attack = make_attack_rows(30, intensity=4.0)
df2_n = pd.DataFrame(rows2_normal, columns=features)
df2_n['Label'] = 'BENIGN'
df2_a = pd.DataFrame(rows2_attack, columns=features)
df2_a['Label'] = 'DoS'
df2 = pd.concat([df2_n, df2_a], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
df2.to_csv(r'd:\nids-vae-project\demo_data\demo2_mixed.csv', index=False)
print("✓ demo2_mixed.csv (70 BENIGN + 30 DoS)")

# ── DEMO 3: 40 binh thuong + 60 tan cong ─────────────────────────
rows3_normal = make_normal_rows(40)
rows3_attack = make_attack_rows(60, intensity=6.0)
df3_n = pd.DataFrame(rows3_normal, columns=features)
df3_n['Label'] = 'BENIGN'
df3_a = pd.DataFrame(rows3_attack, columns=features)
df3_a['Label'] = 'DDoS'
df3 = pd.concat([df3_n, df3_a], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
df3.to_csv(r'd:\nids-vae-project\demo_data\demo3_high_attack.csv', index=False)
print("✓ demo3_high_attack.csv (40 BENIGN + 60 DDoS)")

print(f"\nTat ca file luu tai: d:\\nids-vae-project\\demo_data\\")
print(f"Thu muc: {os.listdir(r'd:\nids-vae-project\demo_data')}")

✓ demo1_normal.csv (100 BENIGN)
✓ demo2_mixed.csv (70 BENIGN + 30 DoS)
✓ demo3_high_attack.csv (40 BENIGN + 60 DDoS)

Tat ca file luu tai: d:\nids-vae-project\demo_data\
Thu muc: ['demo1_normal.csv', 'demo2_mixed.csv', 'demo3_high_attack.csv']


In [9]:
import pandas as pd
import numpy as np
import os

raw_dir = r'd:\nids-vae-project\data\raw'

csv_files = [
    'Monday-WorkingHours.pcap_ISCX.csv',
    'Tuesday-WorkingHours.pcap_ISCX.csv',
    'Wednesday-workingHours.pcap_ISCX.csv',
    'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv',
    'Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv',
    'Friday-WorkingHours-Morning.pcap_ISCX.csv',
    'Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv',
    'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv',
]

# ── Buoc 1: Doc nhan tu tat ca file ──────────────────────────────
print("=== PHAN BO NHAN THEO TUNG NGAY ===\n")
all_labels = []

for fname in csv_files:
    path = os.path.join(raw_dir, fname)
    if not os.path.exists(path):
        print(f"KHONG TIM THAY: {fname}")
        continue

    # Chi doc cot Label de tiet kiem bo nho
    df = pd.read_csv(path, usecols=lambda c: c.strip() in ['Label', ' Label'])
    df.columns = [c.strip() for c in df.columns]
    df['Label'] = df['Label'].str.strip()
    
    counts = df['Label'].value_counts()
    print(f"📁 {fname}")
    for label, count in counts.items():
        tag = '✅ BENIGN' if label == 'BENIGN' else f'🔴 ATTACK'
        print(f"   {tag} | {label:<45} | {count:>8,} rows")
    print()
    
    all_labels.append(df['Label'])

# ── Buoc 2: Tong hop toan bo ─────────────────────────────────────
print("=" * 65)
print("=== TONG HOP TOAN BO DATASET ===\n")
all_series = pd.concat(all_labels)
total = len(all_series)

summary = all_series.value_counts().reset_index()
summary.columns = ['Label', 'Count']
summary['Loai'] = summary['Label'].apply(
    lambda x: 'BENIGN' if x == 'BENIGN' else 'ATTACK'
)
summary['Ti_le_%'] = (summary['Count'] / total * 100).round(2)

print(f"{'Label':<45} {'Loai':<8} {'So luong':>10} {'Ti le':>8}")
print("-" * 75)
for _, row in summary.iterrows():
    icon = '✅' if row['Loai'] == 'BENIGN' else '🔴'
    print(f"{icon} {row['Label']:<43} {row['Loai']:<8} {row['Count']:>10,} {row['Ti_le_%']:>7.2f}%")

print("-" * 75)
print(f"{'TONG CONG':<45} {'':8} {total:>10,} {'100.00%':>8}")

# ── Buoc 3: Phan loai ro rang ─────────────────────────────────────
print("\n=== DANH SACH PHAN LOAI ===")
attack_labels = summary[summary['Loai'] == 'ATTACK']['Label'].tolist()
print(f"\nBINH THUONG (1 loai):")
print(f"  - BENIGN")
print(f"\nTAN CONG ({len(attack_labels)} loai):")
for lb in attack_labels:
    print(f"  - {lb}")

=== PHAN BO NHAN THEO TUNG NGAY ===

📁 Monday-WorkingHours.pcap_ISCX.csv
   ✅ BENIGN | BENIGN                                        |  529,918 rows

📁 Tuesday-WorkingHours.pcap_ISCX.csv
   ✅ BENIGN | BENIGN                                        |  432,074 rows
   🔴 ATTACK | FTP-Patator                                   |    7,938 rows
   🔴 ATTACK | SSH-Patator                                   |    5,897 rows

📁 Wednesday-workingHours.pcap_ISCX.csv
   ✅ BENIGN | BENIGN                                        |  440,031 rows
   🔴 ATTACK | DoS Hulk                                      |  231,073 rows
   🔴 ATTACK | DoS GoldenEye                                 |   10,293 rows
   🔴 ATTACK | DoS slowloris                                 |    5,796 rows
   🔴 ATTACK | DoS Slowhttptest                              |    5,499 rows
   🔴 ATTACK | Heartbleed                                    |       11 rows

📁 Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
   ✅ BENIGN | BENIGN           

In [ ]:
import pandas as pd
import numpy as np
import os
import json

os.makedirs(r'd:\nids-vae-project\demo_data2', exist_ok=True)

raw_dir = r'd:\nids-vae-project\data\raw'

with open(r'd:\nids-vae-project\artifacts\feature_schema\feature_columns.json', 'r') as f:
    schema = json.load(f)
features = schema['feature_columns']

# ── Doc du lieu can thiet ────────────────────────────────────────
print("Dang doc file CSV goc...")

# Monday: chi BENIGN
df_mon = pd.read_csv(
    os.path.join(raw_dir, 'Monday-WorkingHours.pcap_ISCX.csv'),
    usecols=lambda c: c.strip() in features + ['Label']
)
df_mon.columns = [c.strip() for c in df_mon.columns]

# Wednesday: BENIGN + DoS Hulk + DoS GoldenEye
df_wed = pd.read_csv(
    os.path.join(raw_dir, 'Wednesday-workingHours.pcap_ISCX.csv'),
    usecols=lambda c: c.strip() in features + ['Label']
)
df_wed.columns = [c.strip() for c in df_wed.columns]

# Friday DDoS: DDoS + BENIGN
df_fri_ddos = pd.read_csv(
    os.path.join(raw_dir, 'Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv'),
    usecols=lambda c: c.strip() in features + ['Label']
)
df_fri_ddos.columns = [c.strip() for c in df_fri_ddos.columns]

# Friday PortScan
df_fri_ps = pd.read_csv(
    os.path.join(raw_dir, 'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv'),
    usecols=lambda c: c.strip() in features + ['Label']
)
df_fri_ps.columns = [c.strip() for c in df_fri_ps.columns]

print("Doc xong!")

# Xu ly Label
for df in [df_mon, df_wed, df_fri_ddos, df_fri_ps]:
    df['Label'] = df['Label'].str.strip()

# Tach BENIGN va tung loai tan cong
benign_pool  = df_mon[df_mon['Label'] == 'BENIGN']
dos_hulk     = df_wed[df_wed['Label'] == 'DoS Hulk']
dos_golden   = df_wed[df_wed['Label'] == 'DoS GoldenEye']
ddos         = df_fri_ddos[df_fri_ddos['Label'] == 'DDoS']
portscan     = df_fri_ps[df_fri_ps['Label'] == 'PortScan']

print(f"\nPool san sang:")
print(f"  BENIGN     : {len(benign_pool):,}")
print(f"  DoS Hulk   : {len(dos_hulk):,}")
print(f"  DoS Golden : {len(dos_golden):,}")
print(f"  DDoS       : {len(ddos):,}")
print(f"  PortScan   : {len(portscan):,}")

np.random.seed(42)

# ════════════════════════════════════════════════════════════════
# DEMO 1: 200 BENIGN thuần — kiểm tra FPR
# ════════════════════════════════════════════════════════════════
df1 = benign_pool.sample(n=200, random_state=42)[features + ['Label']].copy()
df1.to_csv(r'd:\nids-vae-project\demo_data2\demo1_normal.csv', index=False)

print("\n✓ demo1_normal.csv")
print(f"  200 BENIGN | 0 ATTACK")
print(f"  Muc dich: chung minh FPR thap, he thong khong bao dong nham")

# ════════════════════════════════════════════════════════════════
# DEMO 2: 150 BENIGN + 30 DoS Hulk + 20 DoS GoldenEye — hỗn hợp
# ════════════════════════════════════════════════════════════════
d2_b  = benign_pool.sample(n=150, random_state=1)[features + ['Label']]
d2_a1 = dos_hulk.sample(n=30,  random_state=1)[features + ['Label']]
d2_a2 = dos_golden.sample(n=20, random_state=1)[features + ['Label']]

df2 = pd.concat([d2_b, d2_a1, d2_a2]) \
        .sample(frac=1, random_state=42) \
        .reset_index(drop=True)
df2.to_csv(r'd:\nids-vae-project\demo_data2\demo2_mixed.csv', index=False)

print("\n✓ demo2_mixed.csv")
print(f"  150 BENIGN | 30 DoS Hulk | 20 DoS GoldenEye")
print(f"  Ti le tan cong: 25% | Muc dich: moi truong mang co xuat hien DoS")

# ════════════════════════════════════════════════════════════════
# DEMO 3: 80 BENIGN + 70 DDoS + 50 PortScan — rủi ro cao
# ════════════════════════════════════════════════════════════════
d3_b  = benign_pool.sample(n=80,  random_state=2)[features + ['Label']]
d3_a1 = ddos.sample(n=70,         random_state=2)[features + ['Label']]
d3_a2 = portscan.sample(n=50,     random_state=2)[features + ['Label']]

df3 = pd.concat([d3_b, d3_a1, d3_a2]) \
        .sample(frac=1, random_state=42) \
        .reset_index(drop=True)
df3.to_csv(r'd:\nids-vae-project\demo_data2\demo3_high_attack.csv', index=False)

print("\n✓ demo3_high_attack.csv")
print(f"  80 BENIGN | 70 DDoS | 50 PortScan")
print(f"  Ti le tan cong: 60% | Muc dich: moi truong nguy hiem cao")

# ── Kiem tra chat luong 3 file ────────────────────────────────────
print("\n=== KIEM TRA CHAT LUONG FILE ===")
print(f"{'File':<30} {'Rows':>6} {'NaN':>6} {'BENIGN':>8} {'ATTACK':>8} {'Ti le tan cong':>15}")
print("-" * 80)

for name, path in [
    ('demo1_normal.csv',      r'd:\nids-vae-project\demo_data2\demo1_normal.csv'),
    ('demo2_mixed.csv',       r'd:\nids-vae-project\demo_data2\demo2_mixed.csv'),
    ('demo3_high_attack.csv', r'd:\nids-vae-project\demo_data2\demo3_high_attack.csv'),
]:
    df   = pd.read_csv(path)
    n    = len(df)
    nan  = df[features].isnull().sum().sum()
    nb   = (df['Label'] == 'BENIGN').sum()
    na   = (df['Label'] != 'BENIGN').sum()
    rate = f"{na/n*100:.1f}%"
    print(f"{name:<30} {n:>6} {nan:>6} {nb:>8} {na:>8} {rate:>15}")

print("\nCac loai tan cong trong tung file:")
for name, path in [
    ('Demo1', r'd:\nids-vae-project\demo_data2\demo1_normal.csv'),
    ('Demo2', r'd:\nids-vae-project\demo_data2\demo2_mixed.csv'),
    ('Demo3', r'd:\nids-vae-project\demo_data2\demo3_high_attack.csv'),
]:
    df = pd.read_csv(path)
    attacks = df[df['Label'] != 'BENIGN']['Label'].value_counts()
    print(f"  {name}: {dict(attacks) if len(attacks) > 0 else 'Khong co tan cong'}")